In [1]:
# NORTHSTAR — Solace Browser: Core Pages QA (home, download, start, guide)
# DNA: core_qa = structure(HTML5) x a11y(WCAG) x content(sections) x i18n(data-i18n) x security(CSP) x nav(links)
# Template: solace-cli/notebooks/qa-templates/template-page-qa.ipynb
# Towers: T1(Masters) T6(Hackers) T8(Elders) T9(World) T23(Web)
# Auth: 65537 | Papers: 46,47,49,50
#
# File-based probes — reads HTML/JS/CSS files directly (no running server needed)
# REAL assertions. No mocks. No stubs.

import json, re, hashlib
from pathlib import Path
from datetime import datetime

NORTHSTAR = "solace-browser-core-pages-qa"
NOTEBOOK_PATH = "notebooks/qa/25-core-pages-qa.ipynb"
PROJECT = "solace-browser"
MIN_SCORE = 70

BROWSER = Path("/home/phuc/projects/solace-browser")
WEB = BROWSER / "web"
LOCALES_DIR = BROWSER / "app" / "locales" / "yinyang"

# Core pages under test
CORE_PAGES = ["home", "download", "start", "guide"]

results = []

def record(probe_id, name, passed, detail="", tower_ref=""):
    status = "PASS" if passed else "FAIL"
    results.append({"id": probe_id, "name": name, "status": status,
                    "detail": detail, "tower_ref": tower_ref})
    print(f"  {status}: {name}" + (f" -- {detail}" if detail and not passed else ""))

def read_page(name):
    return (WEB / f"{name}.html").read_text(encoding="utf-8")

# Verify all pages exist
for page in CORE_PAGES:
    assert (WEB / f"{page}.html").exists(), f"{page}.html missing"

print(f"BOOTSTRAP: {len(CORE_PAGES)} core pages under test")
print(f"Pages: {CORE_PAGES}")
print(f"Locales dir exists: {LOCALES_DIR.exists()}")

BOOTSTRAP: 4 core pages under test
Pages: ['home', 'download', 'start', 'guide']
Locales dir exists: True


In [2]:
# P1: HTML Structure — each page has <!DOCTYPE html>, <html lang>, <title>, <meta charset>
print("=== P1: HTML Structure ===")

REQUIRED_STRUCTURE = {
    "doctype": r"<!DOCTYPE html>",
    "html_lang": r'<html\s+lang="[a-z]{2}"',
    "title": r"<title>.+</title>",
    "meta_charset": r'<meta\s+charset="[uU][tT][fF]-8"',
}

for page in CORE_PAGES:
    html = read_page(page)
    for check_name, pattern in REQUIRED_STRUCTURE.items():
        found = bool(re.search(pattern, html, re.IGNORECASE))
        record(f"P1-{page}-{check_name}", f"{page}.html has {check_name}", found,
               detail=f"Pattern {pattern!r} not found in {page}.html" if not found else "",
               tower_ref="T1,T23")

print(f"\nP1 complete: {sum(1 for r in results if r['id'].startswith('P1') and r['status']=='PASS')}/{sum(1 for r in results if r['id'].startswith('P1'))} passed")

=== P1: HTML Structure ===
  PASS: home.html has doctype
  PASS: home.html has html_lang
  PASS: home.html has title
  PASS: home.html has meta_charset
  PASS: download.html has doctype
  PASS: download.html has html_lang
  PASS: download.html has title
  PASS: download.html has meta_charset
  PASS: start.html has doctype
  PASS: start.html has html_lang
  PASS: start.html has title
  PASS: start.html has meta_charset
  PASS: guide.html has doctype
  PASS: guide.html has html_lang
  PASS: guide.html has title
  PASS: guide.html has meta_charset

P1 complete: 16/16 passed


In [3]:
# P2: Content Sections — home has dashboard/apps/schedule widgets; download has platform links;
#     start has getting-started CTA; guide has step-by-step sections
print("=== P2: Content Sections ===")

# home.html: dashboard greeting, app cards section, schedule widget, approval queue
home_html = read_page("home")
home_checks = {
    "greeting_section": r'class="dash-greeting"',
    "app_cards_section": r'class="dash-apps-grid"',
    "schedule_widget": r'dash_schedule_title|Schedule',
    "approval_queue": r'dash_signoff_title|Approval Queue',
    "recent_activity": r'dash_recent|Recent Activity',
}
for name, pattern in home_checks.items():
    found = bool(re.search(pattern, home_html))
    record(f"P2-home-{name}", f"home.html has {name}", found,
           detail=f"Missing {name} in home.html" if not found else "", tower_ref="T1,T23")

# download.html: platform cards for macOS, Linux, Windows
dl_html = read_page("download")
for platform in ["macOS", "Linux", "Windows"]:
    found = bool(re.search(rf"<h3>{platform}</h3>", dl_html))
    record(f"P2-download-{platform.lower()}", f"download.html has {platform} card", found,
           detail=f"Missing {platform} platform card" if not found else "", tower_ref="T1,T23")

# download.html: download links (href with storage.googleapis.com or github)
dl_links = len(re.findall(r'class="platform-link"', dl_html))
record("P2-download-platform_links", f"download.html has platform download links ({dl_links})",
       dl_links >= 3, detail=f"Only {dl_links} platform links found" if dl_links < 3 else "",
       tower_ref="T1,T23")

# start.html: login card with auth buttons (getting-started CTA)
start_html = read_page("start")
start_checks = {
    "login_card": r'class="login-card"',
    "google_auth_btn": r'btn-google|Continue with Google',
    "github_auth_btn": r'btn-github|Sign in with GitHub',
    "email_auth_btn": r'btn-email|Sign in with Email',
}
for name, pattern in start_checks.items():
    found = bool(re.search(pattern, start_html))
    record(f"P2-start-{name}", f"start.html has {name}", found,
           detail=f"Missing {name} in start.html" if not found else "", tower_ref="T1,T23")

# guide.html: step-by-step sections (numbered guide sections)
guide_html = read_page("guide")
guide_sections = re.findall(r'class="guide-num"', guide_html)
record("P2-guide-step_sections", f"guide.html has step-by-step sections ({len(guide_sections)})",
       len(guide_sections) >= 4,
       detail=f"Only {len(guide_sections)} guide sections" if len(guide_sections) < 4 else "",
       tower_ref="T1,T23")

# guide.html: getting started content
has_getting_started = bool(re.search(r'Getting Started Guide', guide_html))
record("P2-guide-getting_started", "guide.html has Getting Started content",
       has_getting_started, tower_ref="T1,T23")

print(f"\nP2 complete: {sum(1 for r in results if r['id'].startswith('P2') and r['status']=='PASS')}/{sum(1 for r in results if r['id'].startswith('P2'))} passed")

=== P2: Content Sections ===
  PASS: home.html has greeting_section
  PASS: home.html has app_cards_section
  PASS: home.html has schedule_widget
  PASS: home.html has approval_queue
  PASS: home.html has recent_activity
  PASS: download.html has macOS card
  PASS: download.html has Linux card
  PASS: download.html has Windows card
  PASS: download.html has platform download links (5)
  PASS: start.html has login_card
  PASS: start.html has google_auth_btn
  PASS: start.html has github_auth_btn
  PASS: start.html has email_auth_btn
  PASS: guide.html has step-by-step sections (6)
  PASS: guide.html has Getting Started content

P2 complete: 15/15 passed


In [4]:
# P3: i18n — data-i18n attributes present, pages load i18n.js or layout.js
print("=== P3: i18n Support ===")

for page in CORE_PAGES:
    html = read_page(page)

    # Check for data-i18n attributes
    i18n_attrs = re.findall(r'data-i18n="([^"]+)"', html)
    record(f"P3-{page}-data_i18n_present", f"{page}.html has data-i18n attributes ({len(i18n_attrs)})",
           len(i18n_attrs) >= 1,
           detail=f"No data-i18n attributes found" if len(i18n_attrs) < 1 else "",
           tower_ref="T9")

    # Check that page loads layout.js (which handles i18n injection) or i18n.js directly
    loads_layout = bool(re.search(r'layout\.js', html))
    loads_i18n = bool(re.search(r'i18n\.js', html))
    record(f"P3-{page}-i18n_script", f"{page}.html loads layout.js or i18n.js",
           loads_layout or loads_i18n,
           detail=f"Neither layout.js nor i18n.js loaded" if not (loads_layout or loads_i18n) else "",
           tower_ref="T9")

print(f"\nP3 complete: {sum(1 for r in results if r['id'].startswith('P3') and r['status']=='PASS')}/{sum(1 for r in results if r['id'].startswith('P3'))} passed")

=== P3: i18n Support ===
  PASS: home.html has data-i18n attributes (33)
  PASS: home.html loads layout.js or i18n.js
  PASS: download.html has data-i18n attributes (15)
  PASS: download.html loads layout.js or i18n.js
  PASS: start.html has data-i18n attributes (11)
  PASS: start.html loads layout.js or i18n.js
  PASS: guide.html has data-i18n attributes (10)
  PASS: guide.html loads layout.js or i18n.js

P3 complete: 8/8 passed


In [5]:
# P4: Navigation — pages load layout.js (header/footer injection), link to each other
print("=== P4: Navigation ===")

for page in CORE_PAGES:
    html = read_page(page)

    # Check layout.js is loaded (provides shared header/footer with navigation)
    has_layout = bool(re.search(r'layout\.js', html))
    record(f"P4-{page}-layout_js", f"{page}.html loads layout.js for header/footer",
           has_layout,
           detail="layout.js not loaded — no shared header/footer" if not has_layout else "",
           tower_ref="T1,T23")

    # Check header-slot exists (layout.js injects nav into this)
    has_header_slot = bool(re.search(r'id="header-slot"', html))
    record(f"P4-{page}-header_slot", f"{page}.html has header-slot for nav injection",
           has_header_slot,
           detail="Missing #header-slot div" if not has_header_slot else "",
           tower_ref="T1,T23")

    # Check footer-slot exists
    has_footer_slot = bool(re.search(r'id="footer-slot"', html))
    record(f"P4-{page}-footer_slot", f"{page}.html has footer-slot",
           has_footer_slot,
           detail="Missing #footer-slot div" if not has_footer_slot else "",
           tower_ref="T1,T23")

# Cross-linking: check that pages reference each other via hrefs
# home links to app-store, download links to github releases, guide links to app-store, start links to guide
cross_links = [
    ("home", r'href="[^"]*app-store', "home links to app-store"),
    ("download", r'href="[^"]*github\.com.*releases', "download links to releases"),
    ("guide", r'href="[^"]*app-store', "guide links to app-store"),
    ("start", r'href="[^"]*guide', "start links to guide"),
]
for page, pattern, desc in cross_links:
    html = read_page(page)
    found = bool(re.search(pattern, html))
    record(f"P4-{page}-crosslink", desc, found,
           detail=f"Cross-link not found: {desc}" if not found else "",
           tower_ref="T1,T23")

print(f"\nP4 complete: {sum(1 for r in results if r['id'].startswith('P4') and r['status']=='PASS')}/{sum(1 for r in results if r['id'].startswith('P4'))} passed")

=== P4: Navigation ===
  PASS: home.html loads layout.js for header/footer
  PASS: home.html has header-slot for nav injection
  PASS: home.html has footer-slot
  PASS: download.html loads layout.js for header/footer
  PASS: download.html has header-slot for nav injection
  PASS: download.html has footer-slot
  PASS: start.html loads layout.js for header/footer
  PASS: start.html has header-slot for nav injection
  PASS: start.html has footer-slot
  PASS: guide.html loads layout.js for header/footer
  PASS: guide.html has header-slot for nav injection
  PASS: guide.html has footer-slot
  PASS: home links to app-store
  PASS: download links to releases
  PASS: guide links to app-store
  PASS: start links to guide

P4 complete: 16/16 passed


In [6]:
# P5: Evidence Summary
print("=== P5: Evidence Summary ===\n")

total = len(results)
passed = sum(1 for r in results if r["status"] == "PASS")
failed = total - passed
score = round((passed / total) * 100, 1) if total > 0 else 0.0

evidence_blob = json.dumps({
    "notebook": "25-core-pages-qa",
    "timestamp": datetime.now().isoformat(),
    "total": total,
    "passed": passed,
    "failed": failed,
    "score": score,
    "results": results,
}, indent=2)

evidence_hash = hashlib.sha256(evidence_blob.encode()).hexdigest()

print(f"  Total probes : {total}")
print(f"  Passed       : {passed}")
print(f"  Failed       : {failed}")
print(f"  Score        : {score}%")
print(f"  Evidence hash: {evidence_hash[:16]}...")
print()

if failed > 0:
    print("FAILURES:")
    for r in results:
        if r["status"] == "FAIL":
            print(f"  FAIL: {r['name']} -- {r['detail']}")

assert score >= MIN_SCORE, f"Score {score}% below minimum {MIN_SCORE}%"
print(f"\nVERDICT: PASS ({score}% >= {MIN_SCORE}% minimum)")

=== P5: Evidence Summary ===

  Total probes : 55
  Passed       : 55
  Failed       : 0
  Score        : 100.0%
  Evidence hash: eef6f77e8757aba9...


VERDICT: PASS (100.0% >= 70% minimum)
